# 7 dpa to 10 dpa 3D Model Alignment

Validate and normalize both stages, reconstruct their 3D models, align them with Spateo, and save the aligned AnnData objects.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import pyvista as pv

pv.global_theme.transparent_background = True

import numpy as np
import spateo as st

import warnings

warnings.filterwarnings("ignore")


## Load and validate data


In [ ]:
stage1_adata = st.read_h5ad("/DATA/User/gaomohan/figures2_\u8865\u5145/data/7dpa1.sc.h5ad")
stage2_adata = st.read_h5ad("/DATA/User/gaomohan/figures2_\u8865\u5145/data/10dpa1.sc.h5ad")

stage1_adata, stage2_adata


### Validate the alignment input contract


In [ ]:
for stage_name, adata in {"7 dpa": stage1_adata, "10 dpa": stage2_adata}.items():
    if "anno" not in adata.obs:
        raise KeyError(f"{stage_name} is missing obs['anno'].")
    if "spatial_3d" not in adata.obsm:
        raise KeyError(f"{stage_name} is missing obsm['spatial_3d'].")
    coordinates = np.asarray(adata.obsm["spatial_3d"])
    if coordinates.shape != (adata.n_obs, 3) or not np.isfinite(coordinates).all():
        raise ValueError(f"{stage_name} requires finite (n_obs, 3) spatial coordinates.")
    if not adata.obs_names.is_unique:
        raise ValueError(f"{stage_name} has duplicated observation identifiers.")
    print(stage_name, adata.shape, adata.obs["anno"].value_counts(dropna=False).to_dict())


In [ ]:
for stage_name, adata in {"stage 1": stage1_adata, "stage 2": stage2_adata}.items():
    if "counts_X" not in adata.layers:
        warnings.warn(
            f"{stage_name} has no count layer; treating X as counts. Verify this assumption."
        )
        adata.layers["counts_X"] = adata.X.copy()
    st.pp.normalize_total(
        adata,
        layer="counts_X",
        out_layer="norm_X",
        target_sum=None,
        size_factor_key="Size_Factor",
        inplace=True,
    )
    st.pp.log1p_layer(
        adata,
        layer="norm_X",
        out_layer="log1p_X",
        set_X=False,
        inplace=True,
    )


## 7 dpa point-cloud and surface reconstruction


## Construct the point-cloud model


In [ ]:
cpo = [
    (296.47267207554813, 858.0952256745953, 4544.382981555063),
    (944.0, 1655.0, 75.0),
    (-0.9898050431631118, 0.043330769787772255, -0.13567763602919042),
]

stage1_pc, plot_cmap = st.tdr.construct_pc(
    adata=stage1_adata.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

st.pl.three_d_plot(
    model=stage1_pc,
    key="tissue",
    model_style="points",
    show_axes=False,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(1600, 800),
    cpo=cpo,
    # filename = f"./figures/10dpa.pdf"
)


## Reconstruct the surface mesh


In [ ]:
cpo = [
    (-3621.284459452889, 1272.4263702089115, 278.8340443397499),
    (944.0, 1655.0, 75.0),
    (0.044336160653487514, 0.0032029812578793612, 0.9990115343526169),
]

stage1_mesh, _, _ = st.tdr.construct_surface(
    pc=stage1_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.08},
    smooth=5000,
    scale_factor=1.02,
)


In [ ]:
cpo = [
    (296.47267207554813, 858.0952256745953, 4544.382981555063),
    (944.0, 1655.0, 75.0),
    (-0.9898050431631118, 0.043330769787772255, -0.13567763602919042),
]

st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_mesh, stage1_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    # opacity=[0.6, 1],
    window_size=(1600, 800),
    cpo=cpo,
    # filename = f"./figures_2/10dpa_pc_mesh_z.pdf",
)


In [ ]:
cpo = [
    (-3621.284459452889, 1272.4263702089115, 278.8340443397499),
    (944.0, 1655.0, 75.0),
    (0.044336160653487514, 0.0032029812578793612, 0.9990115343526169),
]
st.pl.three_d_plot(
    model=st.tdr.collect_models([stage1_mesh, stage1_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    # opacity=[0.6, 1],
    window_size=(1600, 800),
    cpo=cpo,
    # filename = f"./figures_2/10_dpa_pc_mesh_xy.pdf",
)


In [ ]:
cpo = [
    (861.4007529132589, 2028.3753467911467, 4644.846967936437),
    (944.0, 1655.0, 75.0),
    (-0.9817719930373955, 0.18716905163369624, -0.03303785401510178),
]

stage2_pc, plot_cmap = st.tdr.construct_pc(
    adata=stage2_adata.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

st.pl.three_d_plot(
    model=stage2_pc,
    key="tissue",
    model_style="points",
    show_axes=False,
    colormap=plot_cmap,
    jupyter="static",  # False
    window_size=(400, 400),
    cpo=cpo,
    # filename = f"./figures_2/14dpa.pdf"
)


In [ ]:
cpo = [
    (1305.8485686301365, 1893.0791706372074, -4490.316748863963),
    (944.0, 1655.0, 75.0),
    (0.9772773187740892, -0.2011065155392658, 0.06697172252064593),
]

stage2_mesh, _, _ = st.tdr.construct_surface(
    pc=stage2_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.10},
    smooth=5000,
    scale_factor=1.08,
)

st.pl.three_d_plot(
    model=st.tdr.collect_models([stage2_mesh, stage2_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    # opacity=[0.6, 1],
    window_size=(400, 400),
    cpo=cpo,
    # filename = f"./figures_2/14_dpa_pc_mesh_z.pdf",
)


In [ ]:
cpo = [
    (-3469.0966239329905, 2901.617397725647, 58.98915719650984),
    (944.0, 1655.0, 75.0),
    (0.005184768895382603, 0.005511315408809347, -0.9999713713771842),
]

st.pl.three_d_plot(
    model=st.tdr.collect_models([stage2_mesh, stage2_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=False,
    jupyter="static",
    # opacity=[0.6, 1],
    window_size=(400, 400),
    cpo=cpo,
    # filename = f"./figures_2/14_dpa_pc_mesh_z.pdf",
)


In [ ]:
um_stage1_pc_model, um_stage1_mesh_model = stage1_pc.copy(), stage1_mesh.copy()

um_stage1_pc_model.points = um_stage1_pc_model.points / 1000
um_stage1_mesh_model.points = um_stage1_mesh_model.points / 1000


## Calculate morphological features


In [ ]:
morph = st.tdr.model_morphology(model=um_stage1_mesh_model, pc=um_stage1_pc_model)
morph


In [ ]:
um_stage2_pc_model, um_stage2_mesh_model = stage2_pc.copy(), stage2_mesh.copy()

um_stage2_pc_model.points = um_stage2_pc_model.points / 1000
um_stage2_mesh_model.points = um_stage2_mesh_model.points / 1000


In [ ]:
morph = st.tdr.model_morphology(model=um_stage2_mesh_model, pc=um_stage2_pc_model)
morph


## Align developmental stages


In [ ]:
align_samples, align_samples_ref, _, _ = st.align.morpho_align_ref(
    models=[stage1_adata, stage2_adata],
    models_ref=None,
    n_sampling=30000,
    sampling_method="random",
    rep_layer="log1p_X",
    rep_field="layer",
    spatial_key="spatial_3d",
    key_added="3d_align_spatial",
    device="0",
)
align_samples


In [ ]:
stage1_aligned = align_samples[0].copy()
stage2_aligned = align_samples[1].copy()


In [ ]:
stage1_raw_pc, _ = st.tdr.construct_pc(
    adata=stage1_aligned.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

stage1_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage1_aligned.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)


In [ ]:
stage2_raw_pc, _ = st.tdr.construct_pc(
    adata=stage2_aligned.copy(),
    spatial_key="spatial_3d",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

stage2_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage2_aligned.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)


In [ ]:
raw_pair3 = st.tdr.collect_models([stage1_raw_pc.copy(), stage2_raw_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair3,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    show_legend=False,
    show_axes=True,
    jupyter="static",
    window_size=(400, 400),
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


In [ ]:
raw_pair3 = st.tdr.collect_models([stage1_aligned_pc.copy(), stage2_aligned_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair3,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    show_legend=False,
    show_axes=True,
    jupyter="static",
    window_size=(400, 400),
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


In [ ]:
stage1_aligned.write_h5ad(
    "/DATA/User/gaomohan/figures2_\u8865\u5145/data/7dpa_tdr_aligned.h5ad", compression="gzip"
)
stage2_aligned.write_h5ad(
    "/DATA/User/gaomohan/figures2_\u8865\u5145/data/10dpa_tdr_aligned.h5ad", compression="gzip"
)


In [ ]:
stage1_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage1_aligned.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)

stage2_aligned_pc, _ = st.tdr.construct_pc(
    adata=stage2_aligned.copy(),
    spatial_key="3d_align_spatial_nonrigid",
    groupby="anno",
    key_added="tissue",
    colormap="rainbow",
)


## Review the aligned 7 dpa and 10 dpa models


In [ ]:
cpo = [
    (542.3828838487977, 1011.0743234055491, -5251.071106177019),
    (638.7682189941406, 832.677001953125, 190.0),
    (0.9979219551907792, 0.06251028033343381, -0.015628058127874777),
]

raw_pair1 = st.tdr.collect_models([stage1_aligned_pc.copy(), stage2_aligned_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair1,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    show_legend=False,
    show_axes=True,
    jupyter="static",
    cpo=cpo,
    window_size=(400, 400),
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


In [ ]:
cpo = [
    (650.0932471024154, 444.82110374095896, 4673.113818942356),
    (638.7682189941406, 832.677001953125, 190.0),
    (-0.9942329852808687, -0.10702913219951526, -0.006748024904433256),
]

raw_pair1 = st.tdr.collect_models([stage1_aligned_pc.copy(), stage2_aligned_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair1,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    show_legend=False,
    show_axes=True,
    jupyter="static",
    cpo=cpo,
    window_size=(400, 400),
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


In [ ]:
cpo = [
    (-3855.720992217871, 719.4737680098702, 378.7371869912852),
    (638.7682189941406, 832.677001953125, 190.0),
    (0.041963259998284194, -0.0002869920397186735, 0.9991191132422028),
]

raw_pair1 = st.tdr.collect_models([stage1_aligned_pc.copy(), stage2_aligned_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair1,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    show_legend=False,
    show_axes=True,
    jupyter="static",
    cpo=cpo,
    window_size=(400, 400),
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)


## Review the unaligned 7 dpa and 10 dpa models


In [ ]:
cpo = [
    (-38.670576591735845, 1497.9924022172247, 5532.472108831129),
    (831.0, 1466.5, 155.0),
    (-0.9371881807090168, 0.31328320368773016, -0.1534012654005028),
]

raw_pair3 = st.tdr.collect_models([stage1_pc.copy(), stage2_pc.copy()])

st.pl.three_d_plot(
    model=raw_pair3,
    key="groups",
    colormap=["#DC143C", "#0000FF"],
    model_style="points",
    model_size=2,
    show_legend=False,
    show_axes=True,
    jupyter="static",
    cpo=cpo,
    window_size=(400, 400),
    # filename = f"./figures/10dpa_14dpa_nonrigid_aligned.pdf",
)
